## **Week 8: Wednesday: CRUD Patterns in Web Applications**

### **Lecture Objectives**

By the end of this lecture, you will understand:

- HTTP methods and their appropriate use in CRUD operations

- Form handling for both creation and updating of resources

- The concept of idempotency in web operations

- Data validation and error handling in CRUD workflows


## **Theoretical Foundation**

### **1. Understanding CRUD Operations**

CRUD represents the four basic functions of persistent storage:

| Operation | HTTP Method   | Purpose                    | SQL Equivalent |
|-----------|---------------|-----------------------------|----------------|
| Create    | POST          | Make new resource           | INSERT         |
| Read      | GET           | Retrieve existing resource  | SELECT         |
| Update    | PUT / POST     | Modify existing resource     | UPDATE         |
| Delete    | DELETE        | Remove resource              | DELETE         |


### **2. HTTP Methods and Their Semantics**

**Key Concepts:**

- **GET:** Safe, idempotent - should never change server state

- **POST:** Non-idempotent - creates new resources, side effects

- **PUT:** Idempotent - replaces entire resource at specific URI

- **DELETE:** Idempotent - removes resource at specific URI


**Idempotency Explained:**

- ***Idempotent operations:*** Can be repeated without changing result

- ***Example:*** Deleting a resource twice has same effect as once

- ***Non-idempotent:*** Creating same resource twice results in two resources


### **3. Architectural Patterns**

### RESTful Route Design  

_Conventional URL patterns for CRUD:_

| Action   | HTTP Method  | URL Pattern     | Purpose                     |
|----------|---------------|------------------|------------------------------|
| Index    | GET           | /tasks            | List all tasks               |
| Show     | GET           | /tasks/1          | Show specific task           |
| New      | GET           | /tasks/new        | Form to create task          |
| Create   | POST          | /tasks             | Create new task              |
| Edit     | GET           | /tasks/1/edit     | Form to edit task            |
| Update   | PUT / POST     | /tasks/1           | Update specific task         |
| Delete   | DELETE        | /tasks/1           | Remove specific task         |


### **4. Data Layer Setup**

In-memory storage (for demonstration):

```python
class TaskManager:
    def __init__(self):
        self.tasks = []
        self.next_id = 1
    
    def create(self, task_data):
        """Create a new task"""
        task = {
            'id': self.next_id,
            'title': task_data['title'],
            'description': task_data.get('description', ''),
            'completed': task_data.get('completed', False),
            'created_at': datetime.now()
        }
        self.tasks.append(task)
        self.next_id += 1
        return task
    
    def read(self, task_id=None):
        """Read tasks - all or specific"""
        if task_id:
            return next((t for t in self.tasks if t['id'] == task_id), None)
        return self.tasks.copy()
    
    def update(self, task_id, update_data):
        """Update an existing task"""
        task = self.read(task_id)
        if task:
            task.update(update_data)
            return task
        return None
    
    def delete(self, task_id):
        """Delete a task"""
        self.tasks = [t for t in self.tasks if t['id'] != task_id]

### **5. Create Operation Pattern**

**Route Implementation:**



In [ ]:
from Notebooks.Lectures.CSIS1230.Week_8.task_manager import TaskManager
from flask import Flask, request, render_template, redirect, url_for

app = Flask(__name__)

task_manager = TaskManager()

@app.route('/tasks', methods=['GET', 'POST'])
def handle_tasks():
    if request.method == 'POST':
        # CREATE operation
        title = request.form.get('title')
        description = request.form.get('description', '')
        
        # Validation
        errors = []
        if not title:
            errors.append("Title is required")
        if len(title) > 100:
            errors.append("Title must be less than 100 characters")
        
        if errors:
            return render_template('tasks_form.html', errors=errors)
        
        # Business logic
        new_task = task_manager.create({
            'title': title,
            'description': description
        })
        
        # Post-create pattern (redirect to avoid duplicate submissions)
        return redirect(url_for('show_task', task_id=new_task['id']))
    
    else:
        # READ operation (list all)
        tasks = task_manager.read()
        return render_template('tasks_index.html', tasks=tasks)

### **6. Read Operation Patterns**

**Multiple Read Patterns:**

In [ ]:
# Read All (Index)
from flask import abort


@app.route('/tasks')
def list_tasks():
    tasks = task_manager.read()
    return render_template('tasks/index.html', tasks=tasks)

# Read One (Show)
@app.route('/tasks/<int:task_id>')
def show_task(task_id):
    task = task_manager.read(task_id)
    if not task:
        abort(404)  # Resource not found
    return render_template('tasks/show.html', task=task)

### **7. Update Operation Pattern**

**Update Workflow:**

In [ ]:
@app.route('/tasks/<int:task_id>/edit', methods=['GET', 'POST'])
def edit_task(task_id):
    task = task_manager.read(task_id)
    if not task:
        abort(404)
    
    if request.method == 'POST':
        # UPDATE operation
        title = request.form.get('title')
        description = request.form.get('description', '')
        completed = 'completed' in request.form
        
        # Validation (reuse from create)
        errors = []
        if not title:
            errors.append("Title is required")
        
        if errors:
            return render_template('tasks/edit.html', task=task, errors=errors)
        
        # Partial update pattern
        task_manager.update(task_id, {
            'title': title,
            'description': description,
            'completed': completed
        })
        
        return redirect(url_for('show_task', task_id=task_id))
    
    else:
        # Pre-populate form with existing data
        return render_template('tasks/edit.html', task=task)

### **8. Delete Operation Pattern**

**Safe Delete Pattern:**

In [ ]:
@app.route('/tasks/<int:task_id>/delete', methods=['POST'])
def delete_task(task_id):
    task = task_manager.read(task_id)
    if not task:
        abort(404)
    
    # DELETE operation
    task_manager.delete(task_id)
    
    # Post-delete pattern (redirect to index)
    return redirect(url_for('list_tasks'))

### **9. Summary**

This gives an end-to-end framework of what routing patterns would look like for a full project implementation.